# Module 13: Working with NumPy and Pandas

## Integrating Matplotlib with NumPy Arrays and Pandas DataFrames

In this module, you will learn how to bridge Matplotlib with NumPy and Pandas — the two most essential data libraries in the Python scientific stack. Using **CHIRPS rainfall data** for Ethiopia, we explore:

- Plotting raw NumPy arrays (1D, 2D)
- Using `DataFrame.plot()` convenience methods
- Creating grouped plots (by region, season, year)
- Working with time-indexed data (pandas DatetimeIndex)
- Visualizing and handling missing data
- Resampling and aggregation plots

## 1. Setup: Load CHIRPS Data

We load the CHIRPS rainfall dataset using xarray and convert subsets to NumPy arrays / Pandas DataFrames for plotting.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator

# Load CHIRPS dataset
ds = xr.open_dataset('../chirps_1981_2022.nc', engine='netcdf4')
print(ds)

In [ ]:
# Extract key variables and create convenience arrays
precip = ds['precip']

# Ethiopia approximate bounding box
ethiopia = precip.sel(lat=slice(15.0, 3.0), lon=slice(33.0, 48.0))

# Compute spatial mean (Ethiopia-wide average rainfall)
eth_ts = ethiopia.mean(dim=['lat', 'lon'])

# Convert to pandas Series
rainfall_series = eth_ts.to_series()
rainfall_series.name = 'precip'
print(rainfall_series.head())
print(f'Shape: {rainfall_series.shape}')

## 2. Plotting NumPy Arrays (1D)

In [ ]:
# 1D: plot the raw 1D array of Ethiopia-wide monthly rainfall
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(rainfall_series.values, color='steelblue', linewidth=0.8)
ax.set_title('Ethiopia Monthly Rainfall (CHIRPS) — Raw 1D Array', fontsize=13)
ax.set_xlabel('Month index (0 = Jan 1981)')
ax.set_ylabel('Precipitation (mm/month)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Using the time index — proper datetime axis
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(rainfall_series.index, rainfall_series.values, color='steelblue', linewidth=0.8)
ax.set_title('Ethiopia Monthly Rainfall (CHIRPS) — Datetime Index', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Precipitation (mm/month)')
ax.grid(True, alpha=0.3)

# Format date axis
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

## 3. Plotting 2D NumPy Arrays (Maps)

In [ ]:
# Extract a single time slice as 2D array
jan_2000 = precip.sel(time='2000-01-01')
lat_vals = jan_2000['lat'].values
lon_vals = jan_2000['lon'].values
data_2d = jan_2000.values  # shape: (lat, lon)

print(f'2D array shape: {data_2d.shape}')
print(f'Lat range: {lat_vals[0]:.2f} to {lat_vals[-1]:.2f}')
print(f'Lon range: {lon_vals[0]:.2f} to {lon_vals[-1]:.2f}')

In [ ]:
# Visualize the 2D array as a pseudocolor plot
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.pcolormesh(lon_vals, lat_vals, data_2d, shading='auto', cmap='Blues')
ax.set_title('CHIRPS Rainfall — Jan 2000 (2D NumPy Array)', fontsize=12)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
cbar = fig.colorbar(im, ax=ax, label='mm/month')

plt.tight_layout()
plt.show()

## 4. DataFrame.plot() Convenience Methods

Pandas DataFrames provide a `.plot()` wrapper that builds on Matplotlib.

In [ ]:
# Build a multi-region DataFrame
# Define three sub-regions of Ethiopia
regions = {
    'Highlands':   precip.sel(lat=slice(12.0, 10.0), lon=slice(37.0, 40.0)).mean(dim=['lat','lon']).to_series(),
    'Lowlands':    precip.sel(lat=slice(7.0, 5.0),   lon=slice(42.0, 45.0)).mean(dim=['lat','lon']).to_series(),
    'Western':     precip.sel(lat=slice(10.0, 8.0),  lon=slice(34.0, 36.0)).mean(dim=['lat','lon']).to_series(),
}
df_regions = pd.DataFrame(regions)
print(df_regions.head())

In [ ]:
# DataFrame.plot() — line plot
fig, ax = plt.subplots(figsize=(10, 4))
df_regions.plot(ax=ax, alpha=0.85, linewidth=0.7)
ax.set_title('Regional Rainfall Comparison (DataFrame.plot())', fontsize=13)
ax.set_ylabel('Precipitation (mm/month)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot of annual totals using DataFrame.plot.bar()
annual = df_regions.resample('YE').sum()
fig, ax = plt.subplots(figsize=(10, 4))
annual.loc['2000':'2010'].plot.bar(ax=ax, width=0.8)
ax.set_title('Annual Rainfall Totals by Region (DataFrame.plot.bar())', fontsize=13)
ax.set_ylabel('mm/year')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Area plot
fig, ax = plt.subplots(figsize=(10, 4))
annual.loc['2000':'2010'].plot.area(ax=ax, alpha=0.6)
ax.set_title('Cumulative Annual Rainfall by Region', fontsize=13)
ax.set_ylabel('mm/year')
plt.tight_layout()
plt.show()

## 5. Grouped Plots (Region, Season, Year)

Using pandas groupby to create insightful multi-panel views.

In [ ]:
# Group by season (DJF, MAM, JJA, SON)
def season_label(month):
    if month in [12, 1, 2]: return 'DJF'
    if month in [3, 4, 5]: return 'MAM'
    if month in [6, 7, 8]: return 'JJA'
    return 'SON'

df_regions['season'] = df_regions.index.month.map(season_label)
seasonal = df_regions.groupby('season')[['Highlands', 'Lowlands', 'Western']].mean()
print(seasonal)

In [ ]:
# Grouped bar plot
fig, ax = plt.subplots(figsize=(8, 5))
seasonal.plot.bar(ax=ax, width=0.8)
ax.set_title('Mean Seasonal Rainfall by Region', fontsize=13)
ax.set_ylabel('Precipitation (mm/month)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Group by decade
df_regions['decade'] = (df_regions.index.year // 10) * 10
decadal = df_regions.groupby('decade')[['Highlands', 'Lowlands', 'Western']].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

decadal.plot.bar(ax=ax1, width=0.8)
ax1.set_title('Decadal Mean Rainfall', fontsize=12)
ax1.set_ylabel('mm/month')
ax1.grid(True, axis='y', alpha=0.3)

decadal.T.plot.bar(ax=ax2, width=0.8)
ax2.set_title('Regional Comparison per Decade', fontsize=12)
ax2.set_ylabel('mm/month')
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Missing Data Visualization

Real-world datasets often have gaps. We simulate and handle missing values.

In [ ]:
# Create a copy with artificial gaps
rainfall_with_nan = rainfall_series.copy()
np.random.seed(42)
mask = np.random.random(len(rainfall_with_nan)) < 0.05
rainfall_with_nan[mask] = np.nan
print(f'Introduced {mask.sum()} NaN values ({mask.sum()/len(mask)*100:.1f}%)')
print(rainfall_with_nan.isna().sum(), 'total NaN')

In [ ]:
# Visualize missing data with a mask
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# Top: original complete series
ax1.plot(rainfall_series.index, rainfall_series.values, color='steelblue', linewidth=0.7)
ax1.set_title('Original Complete Series', fontsize=11)
ax1.set_ylabel('mm/month')
ax1.grid(True, alpha=0.3)

# Bottom: series with gaps highlighted
ax2.plot(rainfall_with_nan.index, rainfall_with_nan.values, color='lightgray', linewidth=0.7, label='Valid')
nan_mask = rainfall_with_nan.isna()
ax2.scatter(rainfall_with_nan.index[nan_mask], 
           [0]*nan_mask.sum(), 
           color='red', s=15, marker='x', label=f'NaN ({nan_mask.sum()} pts)')
ax2.set_title('Series with Gaps (red = missing)', fontsize=11)
ax2.set_ylabel('mm/month')
ax2.set_xlabel('Date')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Interpolation and plot comparison
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(rainfall_series.index, rainfall_series.values, color='gray', alpha=0.5, label='Original', linewidth=0.7)

interpolated = rainfall_with_nan.interpolate(method='linear', limit_direction='both')
ax.plot(interpolated.index, interpolated.values, color='crimson', linewidth=0.7, label='Linear Interpolation')

ax.fill_between(interpolated.index, rainfall_series.values, interpolated.values, 
                alpha=0.15, color='crimson', label='Interpolation error')
ax.set_title('Missing Data Interpolation (Linear)', fontsize=13)
ax.set_ylabel('mm/month')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Resampling and Aggregation Plots

Pandas resample provides powerful temporal aggregation.

In [ ]:
# Resample to different temporal resolutions
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

# Monthly (raw)
axes[0].plot(rainfall_series.index, rainfall_series.values, color='steelblue', linewidth=0.5)
axes[0].set_title('Monthly (raw)', fontsize=10)
axes[0].set_ylabel('mm')

# Seasonal (3-month)
seasonal_ts = rainfall_series.resample('QE').mean()
axes[1].plot(seasonal_ts.index, seasonal_ts.values, color='forestgreen', linewidth=0.8)
axes[1].set_title('Seasonal (quarterly mean)', fontsize=10)
axes[1].set_ylabel('mm')

# Annual
annual_ts = rainfall_series.resample('YE').sum()
axes[2].bar(annual_ts.index, annual_ts.values, width=200, color='darkorange', alpha=0.7)
axes[2].set_title('Annual (sum)', fontsize=10)
axes[2].set_ylabel('mm/yr')

# Decadal
decadal_ts = rainfall_series.resample('10YE').mean()
axes[3].bar(decadal_ts.index, decadal_ts.values, width=800, color='purple', alpha=0.7)
axes[3].set_title('Decadal (mean)', fontsize=10)
axes[3].set_ylabel('mm/month')
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
# Rolling window for smooth trends
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(rainfall_series.index, rainfall_series.values, color='lightblue', linewidth=0.5, alpha=0.6, label='Monthly')

for window, color, label in [(12, 'blue', '12-month'), (60, 'red', '5-year')]:
    rolled = rainfall_series.rolling(window=window, center=True).mean()
    ax.plot(rolled.index, rolled.values, color=color, linewidth=1.2, label=f'{label} rolling mean')

ax.set_title('Rolling Window Averages (Ethiopia Rainfall)', fontsize=13)
ax.set_ylabel('mm/month')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.YearLocator(10))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 8. Exercises

1. **Annual Cycle**: Compute the monthly climatology (12-month average pattern) of CHIRPS rainfall over Ethiopia. Plot it as a bar chart with 12 bars — one for each month. Use `rainfall_series.groupby(rainfall_series.index.month).mean()`.

2. **Anomaly Plot**: Compute the rainfall anomaly (monthly value minus the climatological mean for that month). Plot anomalies as a colored bar chart (positive blue, negative red).

3. **Regional Box Plot**: Create a box plot comparing the rainfall distributions of the 3 regions from Section 4. Use `df_regions.boxplot()`.

4. **Heatmap**: Create a 2D heatmap (imshow or pcolormesh) of annual rainfall over Ethiopia using a 2D NumPy array extracted from the CHIRPS data. Annotate the plot with coastlines using Cartopy (optional).

### Mini-Project

**Rainfall Variability Analysis**: Using pandas and matplotlib together:
- Extract the Ethiopia-wide time series
- Compute monthly, seasonal, and annual aggregates
- Identify the 5 wettest and 5 driest years on record
- Create a 2x2 figure showing: (a) the full time series with a 5-year rolling mean, (b) annual totals as a bar chart highlighting wet/dry years, (c) monthly climatology, (d) a histogram of monthly rainfall values